# AI Arena — PPO + Self-Play Trainer (NN, Kaggle GPU)

Neural-network combat AI for AshGrid. Uses **PPO** (Stable-Baselines3) on a
3v3 self-play environment, trained on Kaggle GPU notebook.

## What this produces
A `model.onnx` file (~150 KB) you download and the JS game loads with
`onnxruntime-web`. Each AI unit runs the network every frame to pick its
action.

## Quick start
1. Create a new Kaggle notebook with **GPU T4** accelerator.
2. Upload **both** `combat_env.py` and `kaggle_train.py` from this repo's
   `ai_arena/` folder as datasets, OR copy this notebook's code cells into
   a single self-contained notebook. The simplest path:
   - In the Kaggle notebook sidebar: **+ Add Data → Upload → Upload Dataset**
     - Add `combat_env.py` and `kaggle_train.py` together as one dataset
     - Once uploaded, the files are at `/kaggle/input/<dataset-name>/`
3. Edit the **CONFIG** cell (especially `TOTAL_STEPS` and `OPPONENT_MIX`).
4. Run all cells. Training takes ~30 min - 4 hrs depending on `TOTAL_STEPS`.
5. Download `model.onnx` and the checkpoint `.zip` files from Output panel.

---

## CONFIG

In [ ]:
# How many environment steps to train. Time on Kaggle T4 GPU:
#   200_000  ≈ 5-10 min   (sanity test, AI barely competent)
#   1_000_000 ≈ 30-60 min  (first playable AI)
#   4_000_000 ≈ 2-4 hrs    (recommended for first solid AI)
#   16_000_000 ≈ 8 hrs     (max within one Kaggle session, very strong)
TOTAL_STEPS = 100_000   # smoke test (5 min). After verified, bump to 1M+

# Parallel environments (each is one match). More = more GPU memory + faster.
N_ENVS = 16

# Self-play opponent mix (must sum to 1.0):
#   ga: GA-best from kaggle_train.py (strong baseline opponent)
#   self_old: snapshots of past versions of the NN (gradual difficulty)
#   random: random actions (sanity baseline)
OPPONENT_MIX = {'ga': 0.5, 'self_old': 0.4, 'random': 0.1}

# Save a checkpoint every N steps. Each checkpoint = one difficulty tier in JS.
CHECKPOINT_EVERY = 200_000

# How often to add current policy as a new self-play opponent (frozen)
FROZEN_OPP_EVERY = 250_000

# Match length in ticks (45 sec * 60 fps = 2700)
MATCH_TICKS = 2700

# Output paths
import os
IN_KAGGLE = os.path.exists('/kaggle/working')
OUT_DIR = '/kaggle/working/ppo_ckpt' if IN_KAGGLE else './ppo_ckpt'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')
print(f'Total steps: {TOTAL_STEPS:,}')
print(f'Parallel envs: {N_ENVS}')
print(f'Opponent mix: {OPPONENT_MIX}')

## Install dependencies

Kaggle Linux GPU notebook usually has PyTorch + Gymnasium pre-installed.
We just need `stable-baselines3` and `onnx`.

In [ ]:
%pip install -q stable-baselines3 onnx onnxruntime

## Load combat_env and kaggle_train modules

These should be uploaded as a Kaggle dataset (or pasted inline below if you
prefer self-contained).

In [ ]:
import sys, os

# Try to find the dataset directory automatically.
# Replace with your dataset name if needed.
for data_dir in ['/kaggle/input/ai-arena-files', '/kaggle/input', '.', './ai_arena']:
    if os.path.exists(os.path.join(data_dir, 'combat_env.py')):
        sys.path.insert(0, data_dir)
        print(f'Found combat_env.py in {data_dir}')
        break
else:
    print('WARNING: combat_env.py not found in expected paths.')
    print('Either upload it as a Kaggle dataset, or paste its contents into a cell here.')
    print('Searched:', '/kaggle/input/ai-arena-files', '/kaggle/input', '.', './ai_arena')

# Same for kaggle_train (we use its GA simulator as opponent)
for data_dir in ['/kaggle/input/ai-arena-files', '/kaggle/input', '.', './ai_arena']:
    if os.path.exists(os.path.join(data_dir, 'kaggle_train.py')):
        if data_dir not in sys.path:
            sys.path.insert(0, data_dir)
        print(f'Found kaggle_train.py in {data_dir}')
        break

import combat_env
import kaggle_train as ga
print(f'combat_env loaded. OBS_DIM={combat_env.OBS_DIM}, ACTION_DIM={combat_env.ACTION_DIM}')

## Build the self-play opponent pool

Mix of three opponent types:
- **GA-best**: a hard-coded but smart baseline (from `kaggle_train.py`)
- **self_old**: frozen NN snapshots accumulated during training
- **random**: pure noise (curriculum buffer at the bottom)

In [ ]:
import random
import numpy as np
import torch
from typing import List

# Default GA genome (a strong-ish hand-tuned set; replace with a trained one
# if you've already run the GA trainer)
DEFAULT_GA_GENOME = {
    'view_arc': 2.5,
    'view_range': 720,
    'snap_to_threat': 0.30,
    'patrol_scan_speed': 0.04,
    'engage_distance': 280,
    'spread': 0.07,
    'fire_cd_frames': 14,
    'cover_dmg_threshold': 30,
    'peek_duration': 60,
    'hide_duration': 60,
    'flee_hp_pct': 0.35,
    'flank_chance': 0.5,
}

class FrozenNNOpponent:
    """Wrap a frozen NN policy as an opponent. We deepcopy weights so the
    in-flight training policy never updates this snapshot."""
    def __init__(self, policy):
        import copy
        self.policy = copy.deepcopy(policy)
        self.policy.eval()
        for p in self.policy.parameters():
            p.requires_grad_(False)

    def __call__(self, obs_list, env):
        with torch.no_grad():
            obs_arr = np.stack(obs_list).astype(np.float32)
            obs_t = torch.as_tensor(obs_arr)
            actions, _ = self.policy.predict(obs_t, deterministic=False)
            if hasattr(actions, 'tolist'):
                return actions.tolist()
            return list(actions)

frozen_opponents: List = []

def sample_opponent_factory():
    """Return a callable opponent for one new env reset, sampled from the mix."""
    r = random.random()
    cum = 0.0
    for kind, w in OPPONENT_MIX.items():
        cum += w
        if r < cum:
            break
    if kind == 'ga':
        return combat_env.make_ga_opponent(DEFAULT_GA_GENOME)
    elif kind == 'self_old' and frozen_opponents:
        return random.choice(frozen_opponents)
    elif kind == 'random':
        return combat_env.random_opponent
    # Fallback: GA-best (always exists)
    return combat_env.make_ga_opponent(DEFAULT_GA_GENOME)

print('Opponent pool ready.')

## Build the vectorized training environment

Each env is one full 3v3 match. SB3 PPO will sample experience from `N_ENVS`
in parallel. The `SinglePerspectiveEnv` flattens our 3-friendlies-per-match
into SB3's single-agent gym API.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback

def make_env(rank, seed=0):
    """Create one env. Each rollout episode picks a new opponent from the mix."""
    def _init():
        env = combat_env.SinglePerspectiveEnv(
            opponent_policy_fn=sample_opponent_factory,
            squad_size=combat_env.SQUAD_SIZE,
            match_ticks=MATCH_TICKS,
            seed=seed + rank,
        )
        return env
    return _init

# DummyVecEnv runs envs in main process (simpler, fine for our scale).
# SubprocVecEnv would run each in a subprocess (faster but pickling issues).
# On Kaggle GPU notebook with 4 CPU cores, DummyVecEnv is comparable.
vec_env = DummyVecEnv([make_env(i, seed=42) for i in range(N_ENVS)])
print(f'Vec env built with {vec_env.num_envs} envs')
print(f'observation_space: {vec_env.observation_space}')
print(f'action_space: {vec_env.action_space}')

## Build the PPO model

MLP policy, 128x128 hidden, ~50K parameters total. Small enough for browser
inference, big enough to learn tactical behaviour.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# PPO hyperparameters tuned for tactical combat:
#   n_steps:    rollout buffer per env (2048*16 = 32k transitions per update)
#   batch_size: 1024 = good GPU utilisation
#   gamma:      0.99 (medium-horizon, kills happen over ~30 ticks)
#   gae_lambda: 0.95 (standard)
#   ent_coef:   0.01 (encourage exploration; tactical AI needs variation)
#   clip_range: 0.2 (standard)
#   lr:         3e-4 → linear decay to ~1e-5
model = PPO(
    policy='MlpPolicy',
    env=vec_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=1024,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,
    clip_range=0.2,
    policy_kwargs=dict(net_arch=[128, 128]),
    device=device,
    verbose=1,
    seed=42,
)

n_params = sum(p.numel() for p in model.policy.parameters())
print(f'Policy parameters: {n_params:,}')

## Custom callback: log stats + add frozen opponents periodically

In [ ]:
class SelfPlayCallback(BaseCallback):
    """Periodically: (a) snapshot the current policy as a frozen opponent,
    (b) log fresh win-rate against GA-best."""

    def __init__(self, frozen_every=FROZEN_OPP_EVERY, max_pool=6, verbose=0):
        super().__init__(verbose)
        self.frozen_every = frozen_every
        self.max_pool = max_pool
        self.last_frozen_at = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self.last_frozen_at >= self.frozen_every:
            self.last_frozen_at = self.num_timesteps
            new_opp = FrozenNNOpponent(self.model.policy)
            frozen_opponents.append(new_opp)
            if len(frozen_opponents) > self.max_pool:
                frozen_opponents.pop(0)
            print(f'  [step {self.num_timesteps:>9,}] froze new opponent. pool size={len(frozen_opponents)}')
        return True

# CheckpointCallback saves a .zip every N steps for re-loading later.
ckpt_cb = CheckpointCallback(
    save_freq=max(1, CHECKPOINT_EVERY // N_ENVS),
    save_path=OUT_DIR,
    name_prefix='ppo_combat',
    save_replay_buffer=False,
    save_vecnormalize=False,
)
self_play_cb = SelfPlayCallback()
print('Callbacks ready.')

## ▶️ Train

This is the long-running cell. Watch `ep_rew_mean` climb — that's the
average cumulative reward per match per friendly unit. Negative early
(maybe -10 to -50), should rise to positive values (+5 to +50+) by 200k
steps if the system is working.

In [ ]:
from stable_baselines3.common.callbacks import CallbackList

import time
t0 = time.time()
model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=CallbackList([ckpt_cb, self_play_cb]),
    progress_bar=True,
)
elapsed = time.time() - t0
print(f'\nTraining done in {elapsed/60:.1f} min')

# Save final policy
final_path = os.path.join(OUT_DIR, 'ppo_combat_final.zip')
model.save(final_path)
print(f'Final policy saved to {final_path}')

## Export to ONNX (for browser inference)

This converts the SB3 policy to ONNX format that `onnxruntime-web` can load
in the JS game. Output: ~150 KB binary.

In [ ]:
import torch
import torch.nn as nn

class OnnxablePolicy(nn.Module):
    """Wraps the SB3 policy for ONNX export. Input: observation; output: action probs.
    The JS side picks argmax (deterministic) or samples (stochastic)."""
    def __init__(self, sb3_policy):
        super().__init__()
        self.extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, observation):
        latent_pi, _ = self.extractor(observation)
        action_logits = self.action_net(latent_pi)
        action_probs = torch.softmax(action_logits, dim=-1)
        return action_probs

onnx_policy = OnnxablePolicy(model.policy).eval()

import numpy as np
dummy_obs = torch.randn(1, combat_env.OBS_DIM)
onnx_path = os.path.join(OUT_DIR, 'model.onnx')

torch.onnx.export(
    onnx_policy,
    dummy_obs,
    onnx_path,
    input_names=['observation'],
    output_names=['action_probs'],
    dynamic_axes={'observation': {0: 'batch'}, 'action_probs': {0: 'batch'}},
    opset_version=14,
    export_params=True,
)
print(f'ONNX exported to {onnx_path} ({os.path.getsize(onnx_path)} bytes)')

# Sanity check: load ONNX and run inference
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
dummy_in = np.random.randn(1, combat_env.OBS_DIM).astype(np.float32)
out = sess.run(None, {'observation': dummy_in})
print(f'ONNX inference output shape: {out[0].shape}, sample: {out[0][0][:5]}')

## Sanity check: trained NN vs GA-best

Run 10 matches, count NN's win rate. After 1M steps you should see
at least 60-70%. After 4M steps, 80%+.

In [ ]:
ga_opp = combat_env.make_ga_opponent(DEFAULT_GA_GENOME)

wins, losses, draws = 0, 0, 0
for match_idx in range(10):
    test_env = combat_env.CombatEnv(
        opponent_policy=ga_opp, squad_size=combat_env.SQUAD_SIZE,
        match_ticks=MATCH_TICKS, seed=10000 + match_idx,
    )
    obs = test_env.reset(seed=10000 + match_idx)
    while not test_env.done:
        # NN picks actions for friendly team
        obs_batch = np.stack(obs).astype(np.float32)
        with torch.no_grad():
            obs_t = torch.as_tensor(obs_batch).to(device)
            actions, _ = model.policy.predict(obs_t, deterministic=True)
            actions = actions.tolist() if hasattr(actions, 'tolist') else list(actions)
        obs, rewards, done, info = test_env.step(actions)
    if info.get('winner') == 0: wins += 1
    elif info.get('winner') == 1: losses += 1
    else: draws += 1
    print(f'  match {match_idx+1}: kills={info["kills_a"]}-{info["kills_b"]}, '
          f'winner={"NN" if info["winner"]==0 else "GA" if info["winner"]==1 else "draw"}')

print(f'\nNN vs GA-best: {wins}W / {losses}L / {draws}D')
print(f'Win rate: {wins / 10:.0%}')

## What to download

Look in the **Output** panel of this Kaggle notebook. The files you want:

- `model.onnx` — drop this in the JS game's `ai_arena/onnx/` folder
- `ppo_combat_final.zip` — full SB3 model (back up just in case)
- `ppo_combat_*_steps.zip` — checkpoints, one per `CHECKPOINT_EVERY` steps
  (use these as different *difficulty tiers* — earlier = easier)

For each checkpoint you want to ship, re-run the ONNX export cell with that
checkpoint loaded. Save as e.g. `model_easy.onnx`, `model_medium.onnx`,
`model_hard.onnx`.

Then send those `.onnx` files back to me; I'll wire them into the JS game's
difficulty selector.